# MIL-based span localization for inappropriateness

This notebook evaluates Multiple Instance Learning (MIL) as a weakly supervised span localization approach.

Unlike post-hoc methods such as Integrated Gradients or SHAP, MIL trains a new model. Each argument is treated as a bag and automatically generated candidate spans are treated as latent instances. The model assigns a score to each candidate span and aggregates the strongest span scores into an argument-level prediction.

To make the results comparable to post-hoc attribution methods, the selected MIL spans are evaluated with the same perturbation protocol:

1. Select top-ranked MIL spans.
2. Mask the selected spans in the original argument.
3. Run the Ziegenbein document-level classifier on the masked text.
4. Measure the probability drop for the inappropriate class.

Hyperparameters are selected on the validation split and the final configuration is then applied to train, validation, and test.

In [ ]:
!pip install --upgrade --force-reinstall \
    "torch==2.5.1" \
    "transformers==4.46.3" \
    "datasets==3.1.0" \
    "pyarrow==18.1.0" \
    "accelerate>=0.26.0" \
    "sentencepiece" \
    "protobuf" 

In [2]:
import os
import re
import gc
import json
import math
import time
import html
import random
import unicodedata
from pathlib import Path
from datetime import datetime
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn

from datasets import load_dataset
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModel,
    pipeline,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

from tqdm.auto import tqdm
from IPython.display import HTML, display

/opt/conda/lib/python3.11/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/conda/lib/python3.11/site-packages/transformers/data/metrics/__init__.py:19: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 2.4.6)
  from scipy.stats import pearsonr, spearmanr


In [3]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Dataset and model
DATASET_NAME = "timonziegenbein/appropriateness-corpus"
MODEL_NAME = "timonziegenbein/appropriateness-classifier-binary"

# Columns
TEXT_COL = "post_text"
TEXT_NORM_COL = "text_norm"
LABEL_COL = "Inappropriateness"
ID_COL = "post_id"

# Classifier labels
APPROPRIATE_LABEL = "LABEL_0"
INAPPROPRIATE_LABEL = "LABEL_1"

# Device settings
DEVICE = 0 if torch.cuda.is_available() else -1
TORCH_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# General inference settings
CLASSIFIER_MAX_LENGTH = 512
CLASSIFIER_BATCH_SIZE = 32

# MIL input settings
MIL_MAX_LENGTH = 96
MIL_BATCH_SIZE = 2
ENCODER_CHUNK_SIZE = 4
ENCODER_DTYPE = torch.bfloat16

# Training settings
NUM_EPOCHS = 3
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
GRAD_CLIP_NORM = 1.0

# Separate learning rates for encoder and new MIL head
ENCODER_LR = 5e-6
HEAD_LR = 1e-4
HEAD_LR_FROZEN_ENCODER = 1e-3

# Candidate span settings
MIN_WORDS = 2
MAX_CANDIDATES = 80

# MIL initialization
INITIAL_INSTANCE_PROB = 0.02

# Perturbation settings
MASKING_TOP_N_SPANS = 3
ABLATION_MODE = "mask"

# Validation grid.
# Keep this small at first. Expand only after the pipeline works.
POOLING_MODES = [
    "topk_noisy_or",
    "topk_mean",
    "max",
]

TOP_K_VALUES = [
    1,
    3,
    5,
]

SPAN_LENGTH_SETS = [
    (3, 5, 8),
    (5, 10, 15),
    (10, 15, 20),
]

STRIDE_VALUES = [
    2,
    4,
]

FREEZE_ENCODER_VALUES = [
    True,
    # False,  # Activate only after frozen runs look reasonable.
]

# Optional debug limits. Set to None for full runs.
DEBUG_MAX_TRAIN_BAGS = None
DEBUG_MAX_VAL_BAGS = None

print("Device:", TORCH_DEVICE)

Device: cuda


In [ ]:
RUN_NAME = datetime.now().strftime("mil_span_run_%Y%m%d_%H%M%S")
BASE_DIR = Path("results")
OUTPUT_DIR = BASE_DIR / "mil_results" / RUN_NAME
BASE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Saving results to:", OUTPUT_DIR)

Saving results to: mil_results/mil_span_run_20260727_114312


In [5]:
ds = load_dataset(DATASET_NAME)

train_df_raw = ds["train"].to_pandas()
val_df_raw = ds["validation"].to_pandas()
test_df_raw = ds["test"].to_pandas()


def attach_split(df, split_name):
    """Add a split column before creating global IDs."""
    df = df.copy().reset_index(drop=True)
    df["split"] = split_name
    return df


df_all = pd.concat(
    [
        attach_split(train_df_raw, "train"),
        attach_split(val_df_raw, "validation"),
        attach_split(test_df_raw, "test"),
    ],
    ignore_index=True,
)

df_all["global_row_id"] = np.arange(len(df_all))

train_df = df_all[df_all["split"] == "train"].copy().reset_index(drop=True)
val_df = df_all[df_all["split"] == "validation"].copy().reset_index(drop=True)
test_df = df_all[df_all["split"] == "test"].copy().reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

for name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(name)
    display(df[LABEL_COL].value_counts(normalize=True).sort_index())

Train: (1533, 21)
Validation: (220, 21)
Test: (438, 21)
train


Inappropriateness
0    0.456621
1    0.543379
Name: proportion, dtype: float64

validation


Inappropriateness
0    0.436364
1    0.563636
Name: proportion, dtype: float64

test


Inappropriateness
0    0.486301
1    0.513699
Name: proportion, dtype: float64

In [8]:
def normalize_text(text: str) -> str:
    """
    Apply minimal normalization only.
    
    This keeps stylistic signals such as capitalization and punctuation mostly intact,
    while normalizing Unicode and whitespace.
    """
    if text is None:
        return ""
    
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"\s+", " ", text).strip()
    return text


for df in [train_df, val_df, test_df]:
    df[TEXT_NORM_COL] = df[TEXT_COL].apply(normalize_text)

display(train_df[[TEXT_COL, TEXT_NORM_COL]].head())

,post_text,text_norm
0,"people cant be forced to wear school uniforms,...","people cant be forced to wear school uniforms,..."
1,"That form of argument degrades this forum, and...","That form of argument degrades this forum, and..."
2,I wouldnt turn her in becuase she is my wife. ...,I wouldnt turn her in becuase she is my wife. ...
3,No I wouldn't turn in my spouse. Just because ...,No I wouldn't turn in my spouse. Just because ...
4,TV is terrible. Except for Spongebob maybe. Th...,TV is terrible. Except for Spongebob maybe. Th...


In [9]:
pipe = pipeline(
    "text-classification",
    model=MODEL_NAME,
    device=DEVICE,
    top_k=None,
    truncation=True,
    max_length=CLASSIFIER_MAX_LENGTH,
)

classifier_tokenizer = pipe.tokenizer
classifier_model = pipe.model
classifier_model.eval()

MASK_TOKEN = (
    classifier_tokenizer.mask_token
    if classifier_tokenizer.mask_token is not None
    else "[MASK]"
)

print("MASK_TOKEN:", MASK_TOKEN)
print("label2id:", classifier_model.config.label2id)
print("id2label:", classifier_model.config.id2label)

MASK_TOKEN: [MASK]
label2id: {'LABEL_0': 0, 'LABEL_1': 1}
id2label: {0: 'LABEL_0', 1: 'LABEL_1'}


In [10]:
def predict_with_pipeline(texts, batch_size=CLASSIFIER_BATCH_SIZE):
    """
    Predict class probabilities and labels with the Hugging Face pipeline.
    
    Returns one row per input text with:
    - p_inappropriate_original
    - predicted_label
    - predicted_score
    """
    rows = []
    
    for start in tqdm(range(0, len(texts), batch_size), desc="Classifier inference"):
        batch_texts = texts[start:start + batch_size]
        outputs = pipe(batch_texts, batch_size=batch_size)
        
        for output in outputs:
            label_scores = {item["label"]: float(item["score"]) for item in output}
            
            p_inappropriate = label_scores.get(INAPPROPRIATE_LABEL, np.nan)
            best = max(output, key=lambda item: item["score"])
            
            rows.append({
                "p_inappropriate_original": p_inappropriate,
                "predicted_label": best["label"],
                "predicted_score": float(best["score"]),
            })
    
    return pd.DataFrame(rows)


def get_confusion_type(row):
    """Compute confusion type based on gold label and pipeline-predicted label."""
    gold = int(row[LABEL_COL])
    pred_inappropriate = row["predicted_label"] == INAPPROPRIATE_LABEL
    
    if gold == 1 and pred_inappropriate:
        return "TP"
    if gold == 1 and not pred_inappropriate:
        return "FN"
    if gold == 0 and pred_inappropriate:
        return "FP"
    if gold == 0 and not pred_inappropriate:
        return "TN"
    
    return "UNKNOWN"


def add_classifier_predictions(df):
    """Add original Ziegenbein pipeline predictions and confusion flags."""
    df = df.copy()
    pred_df = predict_with_pipeline(df[TEXT_NORM_COL].tolist())
    
    df = pd.concat(
        [df.reset_index(drop=True), pred_df.reset_index(drop=True)],
        axis=1,
    )
    
    df["confusion_type"] = df.apply(get_confusion_type, axis=1)
    df["is_true_positive"] = df["confusion_type"] == "TP"
    df["is_false_negative"] = df["confusion_type"] == "FN"
    df["is_false_positive"] = df["confusion_type"] == "FP"
    df["is_true_negative"] = df["confusion_type"] == "TN"
    
    return df


train_df = add_classifier_predictions(train_df)
val_df = add_classifier_predictions(val_df)
test_df = add_classifier_predictions(test_df)

display(train_df.head())
display(val_df["confusion_type"].value_counts())
display(test_df["confusion_type"].value_counts())

Classifier inference:   0%|          | 0/48 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Classifier inference:   0%|          | 0/7 [00:00<?, ?it/s]

Classifier inference:   0%|          | 0/14 [00:00<?, ?it/s]

,post_id,source_dataset,issue,post_text,Inappropriateness,Toxic Emotions,Excessive Intensity,Emotional Deception,Missing Commitment,Missing Seriousness,...,global_row_id,text_norm,p_inappropriate_original,predicted_label,predicted_score,confusion_type,is_true_positive,is_false_negative,is_false_positive,is_true_negative
0,1,0,Is the school uniform a good or bad idea:,"people cant be forced to wear school uniforms,...",1,0,0,0,1,0,...,0,"people cant be forced to wear school uniforms,...",0.997032,LABEL_1,0.997032,TP,True,False,False,False
1,2,0,Firefox vs internet explorer:,"That form of argument degrades this forum, and...",1,1,0,1,1,1,...,1,"That form of argument degrades this forum, and...",0.998218,LABEL_1,0.998218,TP,True,False,False,False
2,3,0,If your spouse committed murder and he or she ...,I wouldnt turn her in becuase she is my wife. ...,0,0,0,0,0,0,...,2,I wouldnt turn her in becuase she is my wife. ...,0.030317,LABEL_0,0.969683,TN,False,False,False,True
3,4,0,If your spouse committed murder and he or she ...,No I wouldn't turn in my spouse. Just because ...,1,0,0,0,1,0,...,3,No I wouldn't turn in my spouse. Just because ...,0.998705,LABEL_1,0.998705,TP,True,False,False,False
4,5,0,Tv is better than books:,TV is terrible. Except for Spongebob maybe. Th...,1,1,1,0,1,0,...,4,TV is terrible. Except for Spongebob maybe. Th...,0.998405,LABEL_1,0.998405,TP,True,False,False,False


confusion_type
TP    88
TN    66
FN    36
FP    30
Name: count, dtype: int64

confusion_type
TP    186
TN    142
FP     71
FN     39
Name: count, dtype: int64

## Candidate span generation

MIL needs candidate spans before training.  
Each argument is treated as a bag and the generated spans are treated as latent instances.

The parameter `span_lengths` controls the word-window lengths used to generate candidate spans.  

The parameter `stride` controls how densely candidate spans are generated.

In [11]:
def whitespace_token_offsets(text):
    """
    Return whitespace-based tokens with character offsets.
    
    This is intentionally simple and stable for character-level span extraction.
    """
    return [
        {
            "word_index": i,
            "text": match.group(),
            "char_start": int(match.start()),
            "char_end": int(match.end()),
        }
        for i, match in enumerate(re.finditer(r"\S+", str(text)))
    ]


def trim_char_span(text, start, end):
    """Trim whitespace from a character span."""
    while start < end and text[start].isspace():
        start += 1
    while end > start and text[end - 1].isspace():
        end -= 1
    return start, end


def evenly_limit_candidates(candidates, max_candidates):
    """
    Limit candidates while preserving coverage across the full argument.
    
    This avoids keeping only early spans, which would bias localization toward
    the beginning of the argument.
    """
    if max_candidates is None or len(candidates) <= max_candidates:
        return candidates
    
    indices = np.linspace(0, len(candidates) - 1, max_candidates).round().astype(int)
    indices = sorted(set(indices.tolist()))
    return [candidates[i] for i in indices]


def make_candidate_spans(
    text,
    span_lengths=(5, 10, 15),
    stride=4,
    min_words=MIN_WORDS,
    max_candidates=MAX_CANDIDATES,
):
    """
    Create multi-length candidate spans for MIL.
    
    Each candidate span stores:
    - character offsets,
    - whitespace-word offsets,
    - candidate length,
    - text span.
    """
    text = str(text)
    words = whitespace_token_offsets(text)
    
    if len(words) == 0:
        return []
    
    spans = []
    seen = set()
    
    for span_length in span_lengths:
        for word_start_idx in range(0, len(words), stride):
            word_end_exclusive = min(word_start_idx + span_length, len(words))
            window = words[word_start_idx:word_end_exclusive]
            
            if len(window) < min_words:
                continue
            
            char_start = window[0]["char_start"]
            char_end = window[-1]["char_end"]
            char_start, char_end = trim_char_span(text, char_start, char_end)
            
            if char_end <= char_start:
                continue
            
            key = (char_start, char_end)
            if key in seen:
                continue
            
            seen.add(key)
            
            spans.append({
                "span_text": text[char_start:char_end],
                "char_start": int(char_start),
                "char_end": int(char_end),
                "word_start_idx": int(word_start_idx),
                "word_end_idx": int(word_end_exclusive - 1),
                "span_len_words": int(word_end_exclusive - word_start_idx),
                "span_len_chars": int(char_end - char_start),
                "candidate_span_length": int(span_length),
            })
            
            if word_end_exclusive == len(words):
                break
    
    spans = evenly_limit_candidates(spans, max_candidates=max_candidates)
    return spans


example = train_df.iloc[0]
example_spans = make_candidate_spans(
    example[TEXT_NORM_COL],
    span_lengths=(5, 10, 15),
    stride=4,
)

print("Example text:", example[TEXT_NORM_COL])
print("Number of spans:", len(example_spans))
display(pd.DataFrame(example_spans).head())

Example text: people cant be forced to wear school uniforms, i mean each person has theri own wish whether they want to or dont want to wear school uniforms. I think each principal should think once again regarding the uniforms
Number of spans: 25


,span_text,char_start,char_end,word_start_idx,word_end_idx,span_len_words,span_len_chars,candidate_span_length
0,people cant be forced to,0,24,0,4,5,24,5
1,"to wear school uniforms, i",22,48,4,8,5,26,5
2,i mean each person has,47,69,8,12,5,22,5
3,has theri own wish whether,66,92,12,16,5,26,5
4,whether they want to or,85,108,16,20,5,23,5


In [12]:
class AppropriatenessMILDataset(Dataset):
    """
    Dataset for MIL span localization.
    
    One item corresponds to one argument-level bag.
    Candidate spans are latent instances within the bag.
    """
    def __init__(
        self,
        df,
        span_lengths=(5, 10, 15),
        stride=4,
        label_col=LABEL_COL,
        text_col=TEXT_NORM_COL,
        max_candidates=MAX_CANDIDATES,
    ):
        self.rows = []
        self.span_lengths = tuple(span_lengths)
        self.stride = int(stride)
        
        for _, row in df.iterrows():
            text = str(row[text_col])
            issue = str(row["issue"])
            label = int(row[label_col])
            
            candidates = make_candidate_spans(
                text=text,
                span_lengths=self.span_lengths,
                stride=self.stride,
                max_candidates=max_candidates,
            )
            
            if len(candidates) == 0:
                continue
            
            self.rows.append({
                "global_row_id": int(row["global_row_id"]),
                "post_id": row[ID_COL],
                "split": row["split"],
                "issue": issue,
                "text": text,
                "label": label,
                "p_inappropriate_original": float(row["p_inappropriate_original"]),
                "predicted_label_original": row["predicted_label"],
                "predicted_score_original": float(row["predicted_score"]),
                "confusion_type": row["confusion_type"],
                "is_true_positive": bool(row["is_true_positive"]),
                "is_false_negative": bool(row["is_false_negative"]),
                "is_false_positive": bool(row["is_false_positive"]),
                "is_true_negative": bool(row["is_true_negative"]),
                "candidates": candidates,
            })
    
    def __len__(self):
        return len(self.rows)
    
    def __getitem__(self, idx):
        return self.rows[idx]

In [13]:
mil_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def mil_collate_fn(batch, max_length=MIL_MAX_LENGTH):
    """
    Flatten all candidate spans in a batch into one list of encoder inputs.
    
    `bag_ids` stores which flattened span belongs to which original argument.
    """
    flat_inputs = []
    bag_ids = []
    span_metadata = []
    labels = []
    
    bag_metadata = []
    
    for bag_idx, item in enumerate(batch):
        labels.append(item["label"])
        
        bag_metadata.append({
            "global_row_id": item["global_row_id"],
            "post_id": item["post_id"],
            "split": item["split"],
            "issue": item["issue"],
            "text": item["text"],
            "label": item["label"],
            "p_inappropriate_original": item["p_inappropriate_original"],
            "predicted_label_original": item["predicted_label_original"],
            "predicted_score_original": item["predicted_score_original"],
            "confusion_type": item["confusion_type"],
            "is_true_positive": item["is_true_positive"],
            "is_false_negative": item["is_false_negative"],
            "is_false_positive": item["is_false_positive"],
            "is_true_negative": item["is_true_negative"],
            "num_candidate_spans": len(item["candidates"]),
        })
        
        for cand in item["candidates"]:
            # The issue is included because relevance-related inappropriateness
            # can be impossible to judge from a span alone.
            model_input = (
                f"Issue: {item['issue']}\n"
                f"Span: {cand['span_text']}"
            )
            
            flat_inputs.append(model_input)
            bag_ids.append(bag_idx)
            span_metadata.append(cand)
    
    encoded = mil_tokenizer(
        flat_inputs,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
    )
    
    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "bag_ids": torch.tensor(bag_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.float),
        "span_metadata": span_metadata,
        "bag_metadata": bag_metadata,
        "num_bags": len(batch),
    }

In [14]:
class MILPoolingModel(nn.Module):
    """
    MIL model with configurable pooling.
    
    Supported pooling modes:
    - topk_noisy_or: bag is positive if at least one of the top-k spans is positive
    - topk_mean: mean logit over top-k spans
    - max: max-pooling over span logits
    """
    def __init__(
        self,
        model_name,
        pooling_mode="topk_noisy_or",
        top_k=5,
        initial_instance_prob=0.02,
        encoder_chunk_size=4,
        encoder_dtype=torch.bfloat16,
        freeze_encoder=True,
    ):
        super().__init__()
        
        if pooling_mode not in {"topk_noisy_or", "topk_mean", "max"}:
            raise ValueError(f"Unknown pooling mode: {pooling_mode}")
        
        self.encoder = AutoModel.from_pretrained(
            model_name,
            torch_dtype=encoder_dtype,
        )
        
        self.freeze_encoder = bool(freeze_encoder)
        
        if self.freeze_encoder:
            for param in self.encoder.parameters():
                param.requires_grad = False
        else:
            # Useful for memory-saving when fine-tuning.
            self.encoder.gradient_checkpointing_enable()
        
        hidden_size = self.encoder.config.hidden_size
        
        self.instance_classifier = nn.Linear(hidden_size, 1)
        self.pooling_mode = pooling_mode
        self.top_k = int(top_k)
        self.encoder_chunk_size = int(encoder_chunk_size)
        
        initial_bias = math.log(
            initial_instance_prob / (1.0 - initial_instance_prob)
        )
        
        # Small random initialization avoids identical span scores at the beginning.
        nn.init.normal_(self.instance_classifier.weight, mean=0.0, std=0.01)
        nn.init.constant_(self.instance_classifier.bias, initial_bias)
    
    def encode_in_chunks(self, input_ids, attention_mask):
        """
        Encode flattened candidate spans in chunks.
        
        When the encoder is frozen, no gradients are stored for the encoder,
        which substantially speeds up and reduces memory use.
        """
        cls_outputs = []
        
        context = torch.no_grad() if self.freeze_encoder else torch.enable_grad()
        
        with context:
            for start in range(0, input_ids.size(0), self.encoder_chunk_size):
                end = min(start + self.encoder_chunk_size, input_ids.size(0))
                
                outputs = self.encoder(
                    input_ids=input_ids[start:end],
                    attention_mask=attention_mask[start:end],
                    output_hidden_states=False,
                    output_attentions=False,
                    return_dict=True,
                )
                
                cls_outputs.append(outputs.last_hidden_state[:, 0].float())
        
        return torch.cat(cls_outputs, dim=0)
    
    def pool_logits(self, logits):
        """
        Aggregate instance logits into one bag logit.
        
        The output is a logit so that BCEWithLogitsLoss can be used.
        """
        if logits.numel() == 0:
            raise ValueError("Cannot pool an empty set of instance logits.")
        
        if self.pooling_mode == "max":
            return logits.max()
        
        k = min(self.top_k, logits.numel())
        logits_topk, _ = torch.topk(logits, k=k)
        
        if self.pooling_mode == "topk_mean":
            return logits_topk.mean()
        
        if self.pooling_mode == "topk_noisy_or":
            # Stable version of:
            # p_bag = 1 - product_j(1 - sigmoid(logit_j))
            log_p_negative = torch.sum(torch.nn.functional.logsigmoid(-logits_topk))
            log_p_negative = torch.clamp(log_p_negative, max=-1e-6)
            
            log_p_positive = torch.log(-torch.expm1(log_p_negative))
            bag_logit = log_p_positive - log_p_negative
            return bag_logit
        
        raise ValueError(f"Unknown pooling mode: {self.pooling_mode}")
    
    def forward(self, input_ids, attention_mask, bag_ids, num_bags):
        cls = self.encode_in_chunks(input_ids, attention_mask)
        
        instance_logits = self.instance_classifier(cls).squeeze(-1)
        instance_probs = torch.sigmoid(instance_logits)
        
        bag_logits = []
        bag_probs = []
        
        for bag_idx in range(num_bags):
            logits = instance_logits[bag_ids == bag_idx]
            bag_logit = self.pool_logits(logits)
            bag_prob = torch.sigmoid(bag_logit)
            
            bag_logits.append(bag_logit)
            bag_probs.append(bag_prob)
        
        return {
            "bag_logits": torch.stack(bag_logits),
            "bag_probs": torch.stack(bag_probs),
            "instance_logits": instance_logits,
            "instance_probs": instance_probs,
        }

In [15]:
def make_dataloaders(
    train_dataset,
    val_dataset,
    test_dataset=None,
    batch_size=MIL_BATCH_SIZE,
):
    """Create MIL dataloaders."""
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=mil_collate_fn,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=mil_collate_fn,
    )
    
    if test_dataset is None:
        return train_loader, val_loader
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=mil_collate_fn,
    )
    
    return train_loader, val_loader, test_loader


def evaluate_bag_level(model, dataloader, threshold=0.5):
    """
    Evaluate MIL bag-level predictions against document-level labels.
    
    These metrics do not evaluate span correctness directly.
    """
    model.eval()
    
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for batch in dataloader:
            outputs = model(
                input_ids=batch["input_ids"].to(TORCH_DEVICE),
                attention_mask=batch["attention_mask"].to(TORCH_DEVICE),
                bag_ids=batch["bag_ids"].to(TORCH_DEVICE),
                num_bags=batch["num_bags"],
            )
            
            all_labels.extend(batch["labels"].cpu().numpy().tolist())
            all_probs.extend(outputs["bag_probs"].detach().cpu().numpy().tolist())
    
    all_labels = np.array(all_labels, dtype=int)
    all_probs = np.array(all_probs, dtype=float)
    preds = (all_probs >= threshold).astype(int)
    
    metrics = {
        "accuracy": accuracy_score(all_labels, preds),
        "precision": precision_score(all_labels, preds, zero_division=0),
        "recall": recall_score(all_labels, preds, zero_division=0),
        "f1": f1_score(all_labels, preds, zero_division=0),
    }
    
    if len(np.unique(all_labels)) > 1:
        metrics["roc_auc"] = roc_auc_score(all_labels, all_probs)
    else:
        metrics["roc_auc"] = np.nan
    
    metrics["mean_predicted_probability"] = float(np.mean(all_probs))
    metrics["std_predicted_probability"] = float(np.std(all_probs))
    metrics["positive_prediction_rate"] = float(np.mean(preds))
    
    return metrics

In [16]:
def build_optimizer(model, freeze_encoder):
    """Build optimizer with separate learning rates for encoder and MIL head."""
    if freeze_encoder:
        return torch.optim.AdamW(
            model.instance_classifier.parameters(),
            lr=HEAD_LR_FROZEN_ENCODER,
            weight_decay=WEIGHT_DECAY,
        )
    
    return torch.optim.AdamW(
        [
            {"params": model.encoder.parameters(), "lr": ENCODER_LR},
            {"params": model.instance_classifier.parameters(), "lr": HEAD_LR},
        ],
        weight_decay=WEIGHT_DECAY,
    )


def train_one_mil_model(
    config,
    train_dataset,
    val_dataset,
    output_dir,
):
    """
    Train one MIL configuration and return the trained model plus logs.
    
    Hyperparameters are supplied through config.
    """
    train_loader, val_loader = make_dataloaders(
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        batch_size=MIL_BATCH_SIZE,
    )
    
    model = MILPoolingModel(
        model_name=MODEL_NAME,
        pooling_mode=config["pooling_mode"],
        top_k=config["top_k"],
        initial_instance_prob=INITIAL_INSTANCE_PROB,
        encoder_chunk_size=ENCODER_CHUNK_SIZE,
        encoder_dtype=ENCODER_DTYPE,
        freeze_encoder=config["freeze_encoder"],
    ).to(TORCH_DEVICE)
    
    optimizer = build_optimizer(model, freeze_encoder=config["freeze_encoder"])
    
    num_training_steps = NUM_EPOCHS * len(train_loader)
    num_warmup_steps = int(WARMUP_RATIO * num_training_steps)
    
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps,
    )
    
    criterion = nn.BCEWithLogitsLoss()
    training_history = []
    best_val_f1 = -1.0
    best_state_dict = None
    
    start_time = time.time()
    
    for epoch in range(NUM_EPOCHS):
        model.train()
        epoch_start = time.time()
        total_loss = 0.0
        
        for step, batch in enumerate(train_loader):
            optimizer.zero_grad(set_to_none=True)
            
            outputs = model(
                input_ids=batch["input_ids"].to(TORCH_DEVICE),
                attention_mask=batch["attention_mask"].to(TORCH_DEVICE),
                bag_ids=batch["bag_ids"].to(TORCH_DEVICE),
                num_bags=batch["num_bags"],
            )
            
            labels = batch["labels"].to(TORCH_DEVICE).float()
            loss = criterion(outputs["bag_logits"], labels)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            
            optimizer.step()
            scheduler.step()
            
            total_loss += float(loss.item())
            
            if step % 50 == 0:
                print(
                    f"Epoch {epoch + 1}/{NUM_EPOCHS} | "
                    f"Step {step}/{len(train_loader)} | "
                    f"Loss {loss.item():.4f}"
                )
        
        avg_loss = total_loss / max(len(train_loader), 1)
        val_metrics = evaluate_bag_level(model, val_loader)
        epoch_time = (time.time() - epoch_start) / 60
        
        row = {
            "epoch": epoch + 1,
            "train_loss": avg_loss,
            "epoch_time_minutes": epoch_time,
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        
        training_history.append(row)
        
        print(
            f"Epoch {epoch + 1}/{NUM_EPOCHS} finished | "
            f"loss={avg_loss:.4f} | "
            f"val_f1={val_metrics['f1']:.4f} | "
            f"val_auc={val_metrics['roc_auc']:.4f} | "
            f"time={epoch_time:.2f} min"
        )
        
        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state_dict = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
    
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
    
    total_time = (time.time() - start_time) / 60
    
    train_metrics = evaluate_bag_level(model, train_loader)
    val_metrics = evaluate_bag_level(model, val_loader)
    
    result = {
        **config,
        "train_time_minutes": total_time,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }
    
    history_df = pd.DataFrame(training_history)
    
    return model, result, history_df

In [17]:
def collect_mil_outputs(model, dataset, config, threshold=0.5):
    """
    Collect bag-level predictions and span-level scores from a trained MIL model.
    
    Returns:
    - argument_df: one row per argument
    - span_df: one row per candidate span
    """
    model.eval()
    
    argument_rows = []
    span_rows = []
    
    for item_idx, item in enumerate(tqdm(dataset, desc="Collecting MIL outputs")):
        batch = mil_collate_fn([item])
        
        with torch.no_grad():
            outputs = model(
                input_ids=batch["input_ids"].to(TORCH_DEVICE),
                attention_mask=batch["attention_mask"].to(TORCH_DEVICE),
                bag_ids=batch["bag_ids"].to(TORCH_DEVICE),
                num_bags=batch["num_bags"],
            )
        
        bag_prob = float(outputs["bag_probs"].detach().cpu().item())
        pred_label = int(bag_prob >= threshold)
        instance_probs = outputs["instance_probs"].detach().cpu().numpy()
        instance_logits = outputs["instance_logits"].detach().cpu().numpy()
        
        meta = batch["bag_metadata"][0]
        
        argument_rows.append({
            **meta,
            "mil_probability": bag_prob,
            "mil_predicted_label": pred_label,
            "mil_correct": int(pred_label == meta["label"]),
            "mil_confusion_type": (
                "TP" if meta["label"] == 1 and pred_label == 1 else
                "FN" if meta["label"] == 1 and pred_label == 0 else
                "FP" if meta["label"] == 0 and pred_label == 1 else
                "TN"
            ),
            **config,
        })
        
        for prob, logit, span in zip(
            instance_probs,
            instance_logits,
            batch["span_metadata"],
        ):
            span_rows.append({
                "global_row_id": meta["global_row_id"],
                "post_id": meta["post_id"],
                "split": meta["split"],
                "issue": meta["issue"],
                "label": meta["label"],
                "confusion_type": meta["confusion_type"],
                "mil_probability": bag_prob,
                "mil_predicted_label": pred_label,
                "span_score": float(prob),
                "span_logit": float(logit),
                "span_text": span["span_text"],
                "char_start": span["char_start"],
                "char_end": span["char_end"],
                "word_start_idx": span["word_start_idx"],
                "word_end_idx": span["word_end_idx"],
                "span_len_words": span["span_len_words"],
                "span_len_chars": span["span_len_chars"],
                "candidate_span_length": span["candidate_span_length"],
                "text": meta["text"],
                **config,
            })
    
    argument_df = pd.DataFrame(argument_rows)
    span_df = pd.DataFrame(span_rows)
    
    span_df["span_rank"] = (
        span_df
        .groupby(["global_row_id"])["span_score"]
        .rank(method="first", ascending=False)
        .astype(int)
    )
    
    return argument_df, span_df

In [18]:
def merge_overlapping_spans(spans):
    """
    Merge overlapping or directly adjacent character spans.
    
    This prevents repeated masking and produces cleaner span lists.
    """
    if not spans:
        return []
    
    spans = sorted(spans, key=lambda x: (x["char_start"], x["char_end"]))
    merged = [dict(spans[0])]
    
    for span in spans[1:]:
        last = merged[-1]
        
        if int(span["char_start"]) <= int(last["char_end"]):
            last["char_end"] = max(int(last["char_end"]), int(span["char_end"]))
            last["span_text"] = last["text"][last["char_start"]:last["char_end"]] if "text" in last else last["span_text"]
        else:
            merged.append(dict(span))
    
    return merged


def mask_spans_in_text(text, spans, mask_token=MASK_TOKEN):
    """Mask multiple character spans in a text."""
    text = str(text)
    
    if not spans:
        return text
    
    spans = sorted(spans, key=lambda x: int(x["char_start"]))
    
    parts = []
    last_end = 0
    
    for span in spans:
        start = int(span["char_start"])
        end = int(span["char_end"])
        
        if start < last_end:
            continue
        
        parts.append(text[last_end:start])
        parts.append(mask_token)
        last_end = end
    
    parts.append(text[last_end:])
    
    return "".join(parts)


def predict_masked_texts(texts, batch_size=CLASSIFIER_BATCH_SIZE):
    """Predict inappropriate probabilities for masked texts."""
    pred_df = predict_with_pipeline(texts, batch_size=batch_size)
    return pred_df["p_inappropriate_original"].tolist()


def add_mil_perturbation_scores(
    argument_df,
    span_df,
    top_n_spans=MASKING_TOP_N_SPANS,
    mask_token=MASK_TOKEN,
):
    """
    Select top-N MIL spans per argument, mask them, and score the masked text.
    
    The probability drop is computed with the original Ziegenbein classifier,
    making this comparable to IG/SHAP perturbation evaluation.
    """
    argument_df = argument_df.copy()
    span_df = span_df.copy()
    
    top_span_df = (
        span_df
        .sort_values(["global_row_id", "span_score"], ascending=[True, False])
        .groupby("global_row_id")
        .head(top_n_spans)
        .copy()
    )
    
    masked_texts = []
    selected_spans_json = []
    selected_span_texts = []
    selected_char_starts = []
    selected_char_ends = []
    selected_word_starts = []
    selected_word_ends = []
    total_masked_chars = []
    total_masked_words = []
    n_selected_spans = []
    
    for _, row in argument_df.iterrows():
        gid = row["global_row_id"]
        text = row["text"]
        
        spans = top_span_df[top_span_df["global_row_id"] == gid].copy()
        
        span_records = []
        
        for _, span in spans.iterrows():
            span_records.append({
                "span_text": span["span_text"],
                "char_start": int(span["char_start"]),
                "char_end": int(span["char_end"]),
                "word_start_idx": int(span["word_start_idx"]),
                "word_end_idx": int(span["word_end_idx"]),
                "span_len_words": int(span["span_len_words"]),
                "span_len_chars": int(span["span_len_chars"]),
                "span_score": float(span["span_score"]),
                "span_rank": int(span["span_rank"]),
                "candidate_span_length": int(span["candidate_span_length"]),
            })
        
        masked_text = mask_spans_in_text(
            text=text,
            spans=span_records,
            mask_token=mask_token,
        )
        
        masked_texts.append(masked_text)
        selected_spans_json.append(json.dumps(span_records, ensure_ascii=False))
        selected_span_texts.append([s["span_text"] for s in span_records])
        selected_char_starts.append([s["char_start"] for s in span_records])
        selected_char_ends.append([s["char_end"] for s in span_records])
        selected_word_starts.append([s["word_start_idx"] for s in span_records])
        selected_word_ends.append([s["word_end_idx"] for s in span_records])
        total_masked_chars.append(sum(s["span_len_chars"] for s in span_records))
        total_masked_words.append(sum(s["span_len_words"] for s in span_records))
        n_selected_spans.append(len(span_records))
    
    p_masked = predict_masked_texts(masked_texts)
    
    argument_df["attribution_method"] = "mil"
    argument_df["ablation_mode"] = ABLATION_MODE
    argument_df["top_n_masked_spans"] = top_n_spans
    argument_df["masked_text"] = masked_texts
    argument_df["p_inappropriate_masked"] = p_masked
    argument_df["prob_drop"] = (
        argument_df["p_inappropriate_original"] -
        argument_df["p_inappropriate_masked"]
    )
    
    argument_df["selected_spans_json"] = selected_spans_json
    argument_df["selected_span_texts"] = selected_span_texts
    argument_df["selected_span_char_start_indices"] = selected_char_starts
    argument_df["selected_span_char_end_indices"] = selected_char_ends
    argument_df["selected_span_word_start_indices"] = selected_word_starts
    argument_df["selected_span_word_end_indices"] = selected_word_ends
    argument_df["n_selected_spans"] = n_selected_spans
    argument_df["total_masked_chars"] = total_masked_chars
    argument_df["total_masked_words"] = total_masked_words
    
    argument_df["n_words"] = argument_df["text"].apply(lambda x: len(whitespace_token_offsets(x)))
    argument_df["masked_word_ratio"] = (
        argument_df["total_masked_words"] /
        argument_df["n_words"].replace(0, np.nan)
    )
    
    return argument_df, top_span_df

In [19]:
def build_config_id(config):
    """Create a readable config ID for filenames and tables."""
    span_label = "-".join(map(str, config["span_lengths"]))
    return (
        f"pool={config['pooling_mode']}"
        f"__topk={config['top_k']}"
        f"__spans={span_label}"
        f"__stride={config['stride']}"
        f"__freeze={config['freeze_encoder']}"
    )


def subset_dataset(dataset, max_items):
    """Optionally limit a dataset for debugging."""
    if max_items is None:
        return dataset
    
    dataset.rows = dataset.rows[:max_items]
    return dataset


def run_single_validation_config(config):
    """
    Train and evaluate one MIL configuration.
    
    The model is trained on train and evaluated on validation.
    Perturbation is computed on validation true positives from the original
    Ziegenbein classifier to stay comparable with IG/SHAP.
    """
    config = dict(config)
    config["config_id"] = build_config_id(config)
    
    print("\n" + "=" * 100)
    print("Running config:", config["config_id"])
    print("=" * 100)
    
    config_dir = OUTPUT_DIR / "validation_grid" / config["config_id"]
    config_dir.mkdir(parents=True, exist_ok=True)
    
    train_dataset = AppropriatenessMILDataset(
        train_df,
        span_lengths=config["span_lengths"],
        stride=config["stride"],
    )
    
    val_dataset = AppropriatenessMILDataset(
        val_df,
        span_lengths=config["span_lengths"],
        stride=config["stride"],
    )
    
    train_dataset = subset_dataset(train_dataset, DEBUG_MAX_TRAIN_BAGS)
    val_dataset = subset_dataset(val_dataset, DEBUG_MAX_VAL_BAGS)
    
    model, metrics_row, history_df = train_one_mil_model(
        config=config,
        train_dataset=train_dataset,
        val_dataset=val_dataset,
        output_dir=config_dir,
    )
    
    val_argument_df, val_span_df = collect_mil_outputs(
        model=model,
        dataset=val_dataset,
        config=config,
    )
    
    # Perturbation on validation TPs based on the original Ziegenbein classifier.
    val_tp_argument_df = val_argument_df[
        val_argument_df["confusion_type"] == "TP"
    ].copy()
    
    val_tp_span_df = val_span_df[
        val_span_df["global_row_id"].isin(val_tp_argument_df["global_row_id"])
    ].copy()
    
    if len(val_tp_argument_df) > 0:
        val_tp_argument_df, val_top_span_df = add_mil_perturbation_scores(
            argument_df=val_tp_argument_df,
            span_df=val_tp_span_df,
            top_n_spans=MASKING_TOP_N_SPANS,
        )
        
        perturbation_summary = {
            "val_tp_n_arguments": int(val_tp_argument_df["global_row_id"].nunique()),
            "val_tp_mean_prob_drop": float(val_tp_argument_df["prob_drop"].mean()),
            "val_tp_median_prob_drop": float(val_tp_argument_df["prob_drop"].median()),
            "val_tp_positive_drop_rate": float((val_tp_argument_df["prob_drop"] > 0).mean()),
            "val_tp_strong_drop_rate_001": float((val_tp_argument_df["prob_drop"] > 0.01).mean()),
            "val_tp_strong_drop_rate_005": float((val_tp_argument_df["prob_drop"] > 0.05).mean()),
            "val_tp_mean_masked_word_ratio": float(val_tp_argument_df["masked_word_ratio"].mean()),
            "val_tp_mean_total_masked_words": float(val_tp_argument_df["total_masked_words"].mean()),
        }
    else:
        val_top_span_df = pd.DataFrame()
        perturbation_summary = {
            "val_tp_n_arguments": 0,
            "val_tp_mean_prob_drop": np.nan,
            "val_tp_median_prob_drop": np.nan,
            "val_tp_positive_drop_rate": np.nan,
            "val_tp_strong_drop_rate_001": np.nan,
            "val_tp_strong_drop_rate_005": np.nan,
            "val_tp_mean_masked_word_ratio": np.nan,
            "val_tp_mean_total_masked_words": np.nan,
        }
    
    final_row = {
        **metrics_row,
        **perturbation_summary,
    }
    
    history_df.to_csv(config_dir / "training_history.csv", index=False)
    val_argument_df.to_csv(config_dir / "validation_argument_predictions.csv", index=False)
    val_span_df.to_csv(config_dir / "validation_span_scores.csv", index=False)
    val_tp_argument_df.to_csv(config_dir / "validation_tp_perturbation.csv", index=False)
    val_top_span_df.to_csv(config_dir / "validation_tp_top_spans.csv", index=False)
    
    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "config": config,
            "metrics": final_row,
        },
        config_dir / "model_checkpoint.pt",
    )
    
    # Free memory between configs.
    model_cpu_state = {
        k: v.cpu()
        for k, v in model.state_dict().items()
    }
    
    del model
    gc.collect()
    torch.cuda.empty_cache()
    
    return final_row, model_cpu_state

# Load existing Summary to slect config

In [20]:
val_config_summary = pd.read_csv("mil_results/mil_span_run_20260724_091019/mil_validation_config_summary.csv")

display(val_config_summary)

,pooling_mode,top_k,span_lengths,span_lengths_label,stride,freeze_encoder,config_id,train_time_minutes,train_accuracy,train_precision,...,val_std_predicted_probability,val_positive_prediction_rate,val_tp_n_arguments,val_tp_mean_prob_drop,val_tp_median_prob_drop,val_tp_positive_drop_rate,val_tp_strong_drop_rate_001,val_tp_strong_drop_rate_005,val_tp_mean_masked_word_ratio,val_tp_mean_total_masked_words
0,topk_noisy_or,1,"(3, 5, 8)",3-5-8,2,True,pool=topk_noisy_or__topk=1__spans=3-5-8__strid...,39.369830,0.705153,0.830156,...,0.299365,0.340909,88,0.031947,0.000112,0.556818,0.147727,0.068182,0.437449,19.784091
1,topk_noisy_or,3,"(3, 5, 8)",3-5-8,2,True,pool=topk_noisy_or__topk=3__spans=3-5-8__strid...,39.617171,0.669928,0.879350,...,0.321084,0.245455,88,0.056294,0.000105,0.590909,0.204545,0.125000,0.412257,18.988636
2,topk_noisy_or,5,"(3, 5, 8)",3-5-8,2,True,pool=topk_noisy_or__topk=5__spans=3-5-8__strid...,39.556878,0.709067,0.860335,...,0.363772,0.336364,88,0.049984,0.000118,0.602273,0.204545,0.125000,0.410166,18.943182
3,topk_noisy_or,1,"(3, 5, 8)",3-5-8,4,True,pool=topk_noisy_or__topk=1__spans=3-5-8__strid...,31.115266,0.713633,0.705637,...,0.321613,0.559091,88,0.057901,0.000021,0.522727,0.193182,0.136364,0.391577,17.920455
4,topk_noisy_or,3,"(3, 5, 8)",3-5-8,4,True,pool=topk_noisy_or__topk=3__spans=3-5-8__strid...,31.118815,0.711024,0.822848,...,0.356458,0.390909,88,0.042025,0.000080,0.545455,0.193182,0.125000,0.396521,18.272727
5,topk_noisy_or,5,"(3, 5, 8)",3-5-8,4,True,pool=topk_noisy_or__topk=5__spans=3-5-8__strid...,31.132370,0.702544,0.821124,...,0.356559,0.381818,88,0.047337,0.000006,0.511364,0.181818,0.125000,0.396439,18.204545
6,topk_noisy_or,1,"(5, 10, 15)",5-10-15,2,True,pool=topk_noisy_or__topk=1__spans=5-10-15__str...,38.325160,0.706458,0.867562,...,0.324253,0.295455,88,0.083566,0.000267,0.625000,0.204545,0.170455,0.667694,32.500000
7,topk_noisy_or,3,"(5, 10, 15)",5-10-15,2,True,pool=topk_noisy_or__topk=3__spans=5-10-15__str...,38.376781,0.742335,0.847619,...,0.354260,0.377273,88,0.101466,0.000150,0.636364,0.204545,0.181818,0.653329,32.022727
8,topk_noisy_or,5,"(5, 10, 15)",5-10-15,2,True,pool=topk_noisy_or__topk=5__spans=5-10-15__str...,38.363592,0.736464,0.865417,...,0.350296,0.331818,88,0.070183,0.000150,0.613636,0.193182,0.147727,0.661820,32.522727
9,topk_noisy_or,1,"(5, 10, 15)",5-10-15,4,True,pool=topk_noisy_or__topk=1__spans=5-10-15__str...,30.460005,0.748858,0.819088,...,0.368203,0.427273,88,0.074936,0.000246,0.590909,0.204545,0.147727,0.639059,31.659091


Epoch 2/3 | Step 700/767 | Loss 0.4289


Epoch 2/3 | Step 750/767 | Loss 0.9937


Epoch 2/3 finished | loss=0.7492 | val_f1=0.6571 | val_auc=0.7552 | time=10.37 min


Epoch 3/3 | Step 0/767 | Loss 0.0296


...

Epoch 3/3 | Step 700/767 | Loss 1.1322


Epoch 3/3 | Step 750/767 | Loss 0.5915


Epoch 3/3 finished | loss=0.6732 | val_f1=0.6700 | val_auc=0.7785 | time=9.53 min


Classifier inference:   0%|          | 0/3 [00:00<?, ?it/s]

,pooling_mode,top_k,span_lengths,span_lengths_label,stride,freeze_encoder,config_id,train_time_minutes,train_accuracy,train_precision,...,val_std_predicted_probability,val_positive_prediction_rate,val_tp_n_arguments,val_tp_mean_prob_drop,val_tp_median_prob_drop,val_tp_positive_drop_rate,val_tp_strong_drop_rate_001,val_tp_strong_drop_rate_005,val_tp_mean_masked_word_ratio,val_tp_mean_total_masked_words
0,topk_noisy_or,1,"(3, 5, 8)",3-5-8,2,True,pool=topk_noisy_or__topk=1__spans=3-5-8__strid...,39.369830,0.705153,0.830156,...,0.299365,0.340909,88,0.031947,0.000112,0.556818,0.147727,0.068182,0.437449,19.784091
1,topk_noisy_or,3,"(3, 5, 8)",3-5-8,2,True,pool=topk_noisy_or__topk=3__spans=3-5-8__strid...,39.617171,0.669928,0.879350,...,0.321084,0.245455,88,0.056294,0.000105,0.590909,0.204545,0.125000,0.412257,18.988636
2,topk_noisy_or,5,"(3, 5, 8)",3-5-8,2,True,pool=topk_noisy_or__topk=5__spans=3-5-8__strid...,39.556878,0.709067,0.860335,...,0.363772,0.336364,88,0.049984,0.000118,0.602273,0.204545,0.125000,0.410166,18.943182
3,topk_noisy_or,1,"(3, 5, 8)",3-5-8,4,True,pool=topk_noisy_or__topk=1__spans=3-5-8__strid...,31.115266,0.713633,0.705637,...,0.321613,0.559091,88,0.057901,0.000021,0.522727,0.193182,0.136364,0.391577,17.920455
4,topk_noisy_or,3,"(3, 5, 8)",3-5-8,4,True,pool=topk_noisy_or__topk=3__spans=3-5-8__strid...,31.118815,0.711024,0.822848,...,0.356458,0.390909,88,0.042025,0.000080,0.545455,0.193182,0.125000,0.396521,18.272727
5,topk_noisy_or,5,"(3, 5, 8)",3-5-8,4,True,pool=topk_noisy_or__topk=5__spans=3-5-8__strid...,31.132370,0.702544,0.821124,...,0.356559,0.381818,88,0.047337,0.000006,0.511364,0.181818,0.125000,0.396439,18.204545
6,topk_noisy_or,1,"(5, 10, 15)",5-10-15,2,True,pool=topk_noisy_or__topk=1__spans=5-10-15__str...,38.325160,0.706458,0.867562,...,0.324253,0.295455,88,0.083566,0.000267,0.625000,0.204545,0.170455,0.667694,32.500000
7,topk_noisy_or,3,"(5, 10, 15)",5-10-15,2,True,pool=topk_noisy_or__topk=3__spans=5-10-15__str...,38.376781,0.742335,0.847619,...,0.354260,0.377273,88,0.101466,0.000150,0.636364,0.204545,0.181818,0.653329,32.022727
8,topk_noisy_or,5,"(5, 10, 15)",5-10-15,2,True,pool=topk_noisy_or__topk=5__spans=5-10-15__str...,38.363592,0.736464,0.865417,...,0.350296,0.331818,88,0.070183,0.000150,0.613636,0.193182,0.147727,0.661820,32.522727
9,topk_noisy_or,1,"(5, 10, 15)",5-10-15,4,True,pool=topk_noisy_or__topk=1__spans=5-10-15__str...,30.460005,0.748858,0.819088,...,0.368203,0.427273,88,0.074936,0.000246,0.590909,0.204545,0.147727,0.639059,31.659091


In [28]:
MIN_VAL_F1 = 0.70
MIN_VAL_AUC = 0.70
MAX_MASKED_WORD_RATIO = 0.70

candidate_configs = val_config_summary[
    (val_config_summary["val_f1"] >= MIN_VAL_F1) &
    (val_config_summary["val_roc_auc"] >= MIN_VAL_AUC) &
    (val_config_summary["val_tp_mean_masked_word_ratio"] <= MAX_MASKED_WORD_RATIO)
].copy()

candidate_configs = candidate_configs.sort_values(
    ["val_tp_mean_prob_drop", "val_roc_auc", "val_f1"],
    ascending=[False, False, False],
)

display(candidate_configs.head(10))

BEST_CONFIG = candidate_configs.iloc[0].to_dict()

print("Selected config:")
for key, value in BEST_CONFIG.items():
    if key in ["config_id", "pooling_mode", "top_k", "span_lengths", "span_lengths_label", "stride", "freeze_encoder"]:
        print(f"{key}: {value}")

,pooling_mode,top_k,span_lengths,span_lengths_label,stride,freeze_encoder,config_id,train_time_minutes,train_accuracy,train_precision,...,val_std_predicted_probability,val_positive_prediction_rate,val_tp_n_arguments,val_tp_mean_prob_drop,val_tp_median_prob_drop,val_tp_positive_drop_rate,val_tp_strong_drop_rate_001,val_tp_strong_drop_rate_005,val_tp_mean_masked_word_ratio,val_tp_mean_total_masked_words
38,max,1,"(5, 10, 15)",5-10-15,2,True,pool=max__topk=1__spans=5-10-15__stride=2__fre...,38.242323,0.759295,0.780193,...,0.384508,0.527273,88,0.091564,0.000408,0.625000,0.204545,0.170455,0.671015,32.534091
9,topk_noisy_or,1,"(5, 10, 15)",5-10-15,4,True,pool=topk_noisy_or__topk=1__spans=5-10-15__str...,30.460005,0.748858,0.819088,...,0.368203,0.427273,88,0.074936,0.000246,0.590909,0.204545,0.147727,0.639059,31.659091
28,topk_mean,3,"(5, 10, 15)",5-10-15,4,True,pool=topk_mean__topk=3__spans=5-10-15__stride=...,30.196815,0.750163,0.765330,...,0.382091,0.527273,88,0.073830,0.000368,0.647727,0.204545,0.147727,0.657801,32.306818
27,topk_mean,1,"(5, 10, 15)",5-10-15,4,True,pool=topk_mean__topk=1__spans=5-10-15__stride=...,30.213827,0.744292,0.770552,...,0.354700,0.504545,88,0.067464,0.000366,0.647727,0.204545,0.147727,0.648988,32.147727
29,topk_mean,5,"(5, 10, 15)",5-10-15,4,True,pool=topk_mean__topk=5__spans=5-10-15__stride=...,30.227746,0.758643,0.766398,...,0.374860,0.540909,88,0.058224,0.000262,0.602273,0.215909,0.136364,0.626959,31.034091
3,topk_noisy_or,1,"(3, 5, 8)",3-5-8,4,True,pool=topk_noisy_or__topk=1__spans=3-5-8__strid...,31.115266,0.713633,0.705637,...,0.321613,0.559091,88,0.057901,0.000021,0.522727,0.193182,0.136364,0.391577,17.920455
18,topk_mean,1,"(3, 5, 8)",3-5-8,2,True,pool=topk_mean__topk=1__spans=3-5-8__stride=2_...,39.519664,0.736464,0.798331,...,0.345895,0.468182,88,0.053784,0.000183,0.636364,0.193182,0.113636,0.416194,19.056818


Selected config:
pooling_mode: max
top_k: 1
span_lengths: (5, 10, 15)
span_lengths_label: 5-10-15
stride: 2
freeze_encoder: True
config_id: pool=max__topk=1__spans=5-10-15__stride=2__freeze=True


## Selection of the final MIL configuration

The final MIL configuration was selected exclusively on the validation split. The test split was not used for hyperparameter selection.

Since MIL is evaluated both as a document-level classifier and as a span localization method, the selection criterion combines two requirements. First, the model must achieve acceptable bag-level classification performance on the validation split. Therefore, configurations were only considered if they reached a validation F1 score of at least **0.70** and a validation ROC-AUC of at least **0.75**. Second, the selected spans should not explain the prediction by masking excessively large parts of the argument. Therefore, configurations were only considered if their mean masked word ratio on validation true positives was at most **0.70**.

Among all configurations satisfying these constraints, the final configuration was selected by maximizing the mean probability drop on validation true positives. ROC-AUC and F1 were used as secondary tie-breakers. This selection strategy prioritizes spans that have a measurable effect on the original document-level classifier while avoiding configurations that obtain high probability drops mainly by masking most of the argument.

The selected configuration is:

- **Pooling mode:** `max`
- **Top-k:** `1`
- **Span lengths:** `5-10-15`
- **Stride:** `2`
- **Frozen encoder:** `True`

This configuration achieved a validation F1 score of **0.758**, a validation ROC-AUC of **0.766**, and a mean probability drop of **0.092** on validation true positives, while masking on average **0.671** of the words. The resulting spans are therefore interpreted as a balanced trade-off between perturbation effectiveness and localization granularity.

## Final selected MIL configuration

The final MIL configuration is selected on the validation split only.  
The test split is not used for hyperparameter selection.

The selected model is then applied to train, validation, and test to export comparable argument-level and span-level files.

In [38]:
DESIRED_CONFIG = {
    "pooling_mode": "max",
    "top_k": 1,
    "span_lengths": (5, 10, 15),
    "span_lengths_label": "5-10-15",
    "stride": 2,
    "freeze_encoder": True,
    "config_id": "pool=max__topk=1__spans=5-10-15__stride=2__freeze=True",
}

DESIRED_CONFIG_ID = DESIRED_CONFIG["config_id"]
DESIRED_CHECKPOINT_PATH = os.path.join(
    "mil_results",
    "mil_span_run_20260724_091019",
    "validation_grid",
    DESIRED_CONFIG_ID,
    "model_checkpoint.pt",
)

print(DESIRED_CONFIG_ID)
print(DESIRED_CHECKPOINT_PATH)

checkpoint = torch.load(
        DESIRED_CHECKPOINT_PATH,
        map_location="cpu"
    )

desired_model = MILPoolingModel(
    model_name=MODEL_NAME,
    pooling_mode=DESIRED_CONFIG["pooling_mode"],
    top_k=DESIRED_CONFIG["top_k"],
    initial_instance_prob=INITIAL_INSTANCE_PROB,
    encoder_chunk_size=ENCODER_CHUNK_SIZE,
    encoder_dtype=ENCODER_DTYPE,
    freeze_encoder=DESIRED_CONFIG["freeze_encoder"],
)

desired_model.load_state_dict(checkpoint["model_state_dict"])
desired_model = desired_model.to(TORCH_DEVICE)
desired_model.eval()

print("Loaded desired model from checkpoint.")

pool=max__topk=1__spans=5-10-15__stride=2__freeze=True
mil_results/mil_span_run_20260724_091019/validation_grid/pool=max__topk=1__spans=5-10-15__stride=2__freeze=True/model_checkpoint.pt


/tmp/ipykernel_1780200/61876068.py:23: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


Loaded desired model from checkpoint.


In [40]:
def build_dataset_for_desired_config(df):
    """Build a MIL dataset using the final selected configuration."""
    return AppropriatenessMILDataset(
        df,
        span_lengths=DESIRED_CONFIG["span_lengths"],
        stride=DESIRED_CONFIG["stride"],
    )


train_desired_dataset = build_dataset_for_desired_config(train_df)
val_desired_dataset = build_dataset_for_desired_config(val_df)
test_desired_dataset = build_dataset_for_desired_config(test_df)

train_argument_df, train_span_df = collect_mil_outputs(
    model=desired_model,
    dataset=train_desired_dataset,
    config=DESIRED_CONFIG,
)

val_argument_df, val_span_df = collect_mil_outputs(
    model=desired_model,
    dataset=val_desired_dataset,
    config=DESIRED_CONFIG,
)

test_argument_df, test_span_df = collect_mil_outputs(
    model=desired_model,
    dataset=test_desired_dataset,
    config=DESIRED_CONFIG,
)

print("Train:", train_argument_df.shape, train_span_df.shape)
print("Validation:", val_argument_df.shape, val_span_df.shape)
print("Test:", test_argument_df.shape, test_span_df.shape)

Train: (1533, 26) (100474, 27)
Validation: (220, 26) (14571, 27)
Test: (438, 26) (28915, 27)


In [41]:
train_argument_df, train_top_span_df = add_mil_perturbation_scores(
    train_argument_df,
    train_span_df,
    top_n_spans=MASKING_TOP_N_SPANS,
)

val_argument_df, val_top_span_df = add_mil_perturbation_scores(
    val_argument_df,
    val_span_df,
    top_n_spans=MASKING_TOP_N_SPANS,
)

test_argument_df, test_top_span_df = add_mil_perturbation_scores(
    test_argument_df,
    test_span_df,
    top_n_spans=MASKING_TOP_N_SPANS,
)

final_argument_df = pd.concat(
    [train_argument_df, val_argument_df, test_argument_df],
    ignore_index=True,
)

final_span_df = pd.concat(
    [train_span_df, val_span_df, test_span_df],
    ignore_index=True,
)

final_top_span_df = pd.concat(
    [train_top_span_df, val_top_span_df, test_top_span_df],
    ignore_index=True,
)

display(final_argument_df.head())
display(final_span_df.head())
display(final_top_span_df.head())

Classifier inference:   0%|          | 0/48 [00:00<?, ?it/s]

Classifier inference:   0%|          | 0/7 [00:00<?, ?it/s]

Classifier inference:   0%|          | 0/14 [00:00<?, ?it/s]

,global_row_id,post_id,split,issue,text,label,p_inappropriate_original,predicted_label_original,predicted_score_original,confusion_type,...,selected_span_texts,selected_span_char_start_indices,selected_span_char_end_indices,selected_span_word_start_indices,selected_span_word_end_indices,n_selected_spans,total_masked_chars,total_masked_words,n_words,masked_word_ratio
0,0,1,train,Is the school uniform a good or bad idea:,"people cant be forced to wear school uniforms,...",1,0.997032,LABEL_1,0.997032,TP,...,[has theri own wish whether they want to or do...,"[66, 12, 0]","[113, 65, 53]","[12, 2, 0]","[21, 11, 9]",3,153,30,38,0.789474
1,1,2,train,Firefox vs internet explorer:,"That form of argument degrades this forum, and...",1,0.998218,LABEL_1,0.998218,TP,...,"[lowest common denominator. This word, ""indisp...","[87, 58, 80]","[155, 149, 169]","[16, 10, 14]","[25, 24, 28]",3,248,40,63,0.634921
2,2,3,train,If your spouse committed murder and he or she ...,I wouldnt turn her in becuase she is my wife. ...,0,0.030317,LABEL_0,0.969683,TN,...,[trusted me by telling me what she did then I ...,"[97, 90, 0]","[166, 158, 45]","[22, 20, 0]","[36, 34, 9]",3,182,40,37,1.081081
3,3,4,train,If your spouse committed murder and he or she ...,No I wouldn't turn in my spouse. Just because ...,1,0.998705,LABEL_1,0.998705,TP,...,[in my spouse. Just because the girl that i ma...,"[19, 38, 25]","[69, 61, 84]","[4, 8, 6]","[13, 12, 15]",3,132,25,25,1.000000
4,4,5,train,Tv is better than books:,TV is terrible. Except for Spongebob maybe. Th...,1,0.998405,LABEL_1,0.998405,TP,...,"[terrible. Except for Spongebob maybe., for Sp...","[6, 23, 102]","[43, 53, 123]","[2, 4, 16]","[6, 8, 20]",3,88,15,27,0.555556


,global_row_id,post_id,split,issue,label,confusion_type,mil_probability,mil_predicted_label,span_score,span_logit,...,candidate_span_length,text,pooling_mode,top_k,span_lengths,span_lengths_label,stride,freeze_encoder,config_id,span_rank
0,0,1,train,Is the school uniform a good or bad idea:,1,TP,0.061119,0,0.001785,-6.326402,...,5,"people cant be forced to wear school uniforms,...",max,1,"(5, 10, 15)",5-10-15,2,True,pool=max__topk=1__spans=5-10-15__stride=2__fre...,20
1,0,1,train,Is the school uniform a good or bad idea:,1,TP,0.061119,0,0.000734,-7.215740,...,5,"people cant be forced to wear school uniforms,...",max,1,"(5, 10, 15)",5-10-15,2,True,pool=max__topk=1__spans=5-10-15__stride=2__fre...,26
2,0,1,train,Is the school uniform a good or bad idea:,1,TP,0.061119,0,0.000078,-9.461362,...,5,"people cant be forced to wear school uniforms,...",max,1,"(5, 10, 15)",5-10-15,2,True,pool=max__topk=1__spans=5-10-15__stride=2__fre...,43
3,0,1,train,Is the school uniform a good or bad idea:,1,TP,0.061119,0,0.008808,-4.723269,...,5,"people cant be forced to wear school uniforms,...",max,1,"(5, 10, 15)",5-10-15,2,True,pool=max__topk=1__spans=5-10-15__stride=2__fre...,7
4,0,1,train,Is the school uniform a good or bad idea:,1,TP,0.061119,0,0.010050,-4.590069,...,5,"people cant be forced to wear school uniforms,...",max,1,"(5, 10, 15)",5-10-15,2,True,pool=max__topk=1__spans=5-10-15__stride=2__fre...,6


,global_row_id,post_id,split,issue,label,confusion_type,mil_probability,mil_predicted_label,span_score,span_logit,...,candidate_span_length,text,pooling_mode,top_k,span_lengths,span_lengths_label,stride,freeze_encoder,config_id,span_rank
0,0,1,train,Is the school uniform a good or bad idea:,1,TP,0.061119,0,0.061119,-2.731864,...,10,"people cant be forced to wear school uniforms,...",max,1,"(5, 10, 15)",5-10-15,2,True,pool=max__topk=1__spans=5-10-15__stride=2__fre...,1
1,0,1,train,Is the school uniform a good or bad idea:,1,TP,0.061119,0,0.041546,-3.138513,...,10,"people cant be forced to wear school uniforms,...",max,1,"(5, 10, 15)",5-10-15,2,True,pool=max__topk=1__spans=5-10-15__stride=2__fre...,2
2,0,1,train,Is the school uniform a good or bad idea:,1,TP,0.061119,0,0.036896,-3.262045,...,10,"people cant be forced to wear school uniforms,...",max,1,"(5, 10, 15)",5-10-15,2,True,pool=max__topk=1__spans=5-10-15__stride=2__fre...,3
3,1,2,train,Firefox vs internet explorer:,1,TP,0.869586,1,0.869586,1.897302,...,10,"That form of argument degrades this forum, and...",max,1,"(5, 10, 15)",5-10-15,2,True,pool=max__topk=1__spans=5-10-15__stride=2__fre...,1
4,1,2,train,Firefox vs internet explorer:,1,TP,0.869586,1,0.853104,1.759160,...,15,"That form of argument degrades this forum, and...",max,1,"(5, 10, 15)",5-10-15,2,True,pool=max__topk=1__spans=5-10-15__stride=2__fre...,2


In [2]:
overall_summary = (
    final_argument_df
    .groupby([
        "split",
        "attribution_method",
        "pooling_mode",
        "top_k",
        "span_lengths_label",
        "stride",
        "freeze_encoder",
    ])
    .agg(
        n_arguments=("global_row_id", "nunique"),
        mean_prob_drop=("prob_drop", "mean"),
        median_prob_drop=("prob_drop", "median"),
        std_prob_drop=("prob_drop", "std"),
        positive_drop_rate=("prob_drop", lambda x: (x > 0).mean()),
        strong_drop_rate_001=("prob_drop", lambda x: (x > 0.01).mean()),
        strong_drop_rate_005=("prob_drop", lambda x: (x > 0.05).mean()),
        mean_p_original=("p_inappropriate_original", "mean"),
        mean_p_masked=("p_inappropriate_masked", "mean"),
        mean_mil_probability=("mil_probability", "mean"),
        std_mil_probability=("mil_probability", "std"),
        mean_n_selected_spans=("n_selected_spans", "mean"),
        mean_total_masked_words=("total_masked_words", "mean"),
        mean_total_masked_chars=("total_masked_chars", "mean"),
        mean_masked_word_ratio=("masked_word_ratio", "mean"),
    )
    .reset_index()
)

confusion_summary = (
    final_argument_df
    .groupby(["split", "confusion_type"])
    .agg(
        n_arguments=("global_row_id", "nunique"),
        mean_prob_drop=("prob_drop", "mean"),
        median_prob_drop=("prob_drop", "median"),
        std_prob_drop=("prob_drop", "std"),
        positive_drop_rate=("prob_drop", lambda x: (x > 0).mean()),
        mean_p_original=("p_inappropriate_original", "mean"),
        mean_p_masked=("p_inappropriate_masked", "mean"),
        mean_mil_probability=("mil_probability", "mean"),
        mean_total_masked_words=("total_masked_words", "mean"),
        mean_masked_word_ratio=("masked_word_ratio", "mean"),
    )
    .reset_index()
    .sort_values(["split", "confusion_type"])
)

test_summary = overall_summary[
    overall_summary["split"] == "test"
].copy()

display(overall_summary)
display(confusion_summary)
display(test_summary)

,split,attribution_method,pooling_mode,top_k,span_lengths_label,stride,freeze_encoder,n_arguments,mean_prob_drop,median_prob_drop,...,strong_drop_rate_001,strong_drop_rate_005,mean_p_original,mean_p_masked,mean_mil_probability,std_mil_probability,mean_n_selected_spans,mean_total_masked_words,mean_total_masked_chars,mean_masked_word_ratio
0,test,mil,max,1,5-10-15,2,True,438,-0.006984,0.000060,...,0.189498,0.130137,0.590804,0.597789,0.546126,0.385445,3.0,30.381279,164.874429,0.553915
1,train,mil,max,1,5-10-15,2,True,1533,-0.029605,-0.000022,...,0.155251,0.098500,0.552141,0.581746,0.524619,0.394119,3.0,30.585780,166.688845,0.568406
2,validation,mil,max,1,5-10-15,2,True,220,-0.010937,0.000006,...,0.181818,0.122727,0.542075,0.553012,0.502037,0.385373,3.0,29.754545,160.918182,0.531068


,split,confusion_type,n_arguments,mean_prob_drop,median_prob_drop,std_prob_drop,positive_drop_rate,mean_p_original,mean_p_masked,mean_mil_probability,mean_total_masked_words,mean_masked_word_ratio
0,test,FN,39,-0.182820,-0.001893,0.364982,0.435897,0.049638,0.232457,0.486211,28.205128,0.406339
1,test,FP,71,0.130435,0.000513,0.325707,0.605634,0.971585,0.841150,0.533580,31.239437,0.647548
2,test,TN,142,-0.129318,-0.001311,0.315043,0.408451,0.037216,0.166533,0.339491,27.654930,0.353890
3,test,TP,186,0.070823,0.000112,0.229214,0.602151,0.981555,0.910732,0.721231,32.591398,0.701824
4,train,FN,48,-0.210893,-0.003534,0.406408,0.416667,0.115297,0.326190,0.382080,28.645833,0.488078
5,train,FP,48,0.082692,0.000132,0.319487,0.541667,0.915789,0.833097,0.361602,31.645833,0.708483
6,train,TN,652,-0.148090,-0.001845,0.314790,0.337423,0.030158,0.178248,0.277089,27.773006,0.419322
7,train,TP,785,0.073024,0.000144,0.224571,0.620382,0.990161,0.917137,0.748896,32.975796,0.688578
8,validation,FN,36,-0.202391,-0.005998,0.357832,0.305556,0.040581,0.242972,0.468840,28.805556,0.407293
9,validation,FP,30,0.175948,0.000403,0.353247,0.566667,0.968823,0.792874,0.395860,27.366667,0.546571


,split,attribution_method,pooling_mode,top_k,span_lengths_label,stride,freeze_encoder,n_arguments,mean_prob_drop,median_prob_drop,...,strong_drop_rate_001,strong_drop_rate_005,mean_p_original,mean_p_masked,mean_mil_probability,std_mil_probability,mean_n_selected_spans,mean_total_masked_words,mean_total_masked_chars,mean_masked_word_ratio
0,test,mil,max,1,5-10-15,2,True,438,-0.006984,0.00006,...,0.189498,0.130137,0.590804,0.597789,0.546126,0.385445,3.0,30.381279,164.874429,0.553915


In [ ]:
final_output_dir = OUTPUT_DIR / DESIRED_CONFIG_ID
final_output_dir.mkdir(parents=True, exist_ok=True)

argument_csv_path = final_output_dir / (
    f"mil_final_all_splits_{DESIRED_CONFIG_ID}_argument_level.csv"
)

span_csv_path = final_output_dir / (
    f"mil_final_all_splits_{DESIRED_CONFIG_ID}_span_level.csv"
)

top_span_csv_path = final_output_dir / (
    f"mil_final_all_splits_{DESIRED_CONFIG_ID}_top_spans.csv"
)

overall_summary_csv_path = final_output_dir / (
    f"mil_final_all_splits_{DESIRED_CONFIG_ID}_overall_summary.csv"
)

confusion_summary_csv_path = final_output_dir / (
    f"mil_final_all_splits_{DESIRED_CONFIG_ID}_confusion_summary.csv"
)

test_summary_csv_path = final_output_dir / (
    f"mil_final_test_{DESIRED_CONFIG_ID}_summary.csv"
)

final_argument_df.to_csv(argument_csv_path, index=False, encoding="utf-8")
final_span_df.to_csv(span_csv_path, index=False, encoding="utf-8")
final_top_span_df.to_csv(top_span_csv_path, index=False, encoding="utf-8")
overall_summary.to_csv(overall_summary_csv_path, index=False, encoding="utf-8")
confusion_summary.to_csv(confusion_summary_csv_path, index=False, encoding="utf-8")
test_summary.to_csv(test_summary_csv_path, index=False, encoding="utf-8")

with open(final_output_dir / "selected_config.json", "w", encoding="utf-8") as f:
    json.dump(DESIRED_CONFIG, f, indent=2, ensure_ascii=False)

torch.save(
    {
        "model_state_dict": desired_model.state_dict(),
        "config": DESIRED_CONFIG,
    },
    final_output_dir / "selected_mil_model_checkpoint.pt",
)

print("Saved:")
print(argument_csv_path)
print(span_csv_path)
print(top_span_csv_path)
print(overall_summary_csv_path)
print(confusion_summary_csv_path)
print(test_summary_csv_path)

Saved:
results/mil_results/mil_span_run_20260727_114312/pool=max__topk=1__spans=5-10-15__stride=2__freeze=True/mil_final_all_splits_pool=max__topk=1__spans=5-10-15__stride=2__freeze=True_argument_level.csv
results/mil_results/mil_span_run_20260727_114312/pool=max__topk=1__spans=5-10-15__stride=2__freeze=True/mil_final_all_splits_pool=max__topk=1__spans=5-10-15__stride=2__freeze=True_span_level.csv
results/mil_results/mil_span_run_20260727_114312/pool=max__topk=1__spans=5-10-15__stride=2__freeze=True/mil_final_all_splits_pool=max__topk=1__spans=5-10-15__stride=2__freeze=True_top_spans.csv
results/mil_results/mil_span_run_20260727_114312/pool=max__topk=1__spans=5-10-15__stride=2__freeze=True/mil_final_all_splits_pool=max__topk=1__spans=5-10-15__stride=2__freeze=True_overall_summary.csv
results/mil_results/mil_span_run_20260727_114312/pool=max__topk=1__spans=5-10-15__stride=2__freeze=True/mil_final_all_splits_pool=max__topk=1__spans=5-10-15__stride=2__freeze=True_confusion_summary.csv
res

In [44]:
def highlight_spans(text, spans):
    """Highlight multiple character spans in one text."""
    text = str(text)
    
    if not spans:
        return html.escape(text)
    
    spans = sorted(spans, key=lambda x: int(x["char_start"]))
    
    parts = []
    last_end = 0
    
    for span in spans:
        start = int(span["char_start"])
        end = int(span["char_end"])
        
        if start < last_end:
            continue
        
        parts.append(html.escape(text[last_end:start]))
        parts.append(
            "<mark style='background-color:#ffe58a; padding:2px 4px; border-radius:4px;'>"
            + html.escape(text[start:end])
            + "</mark>"
        )
        last_end = end
    
    parts.append(html.escape(text[last_end:]))
    return "".join(parts)


def show_mil_example(result_df, index=0):
    """Display one MIL result with highlighted selected spans."""
    row = result_df.iloc[index]
    
    spans = json.loads(row["selected_spans_json"])
    marked = highlight_spans(row["text"], spans)
    
    display(HTML(f"""
    <div style="
        border:1px solid #ccc;
        border-radius:8px;
        padding:14px;
        margin:12px 0;
        font-family:Arial, sans-serif;
        line-height:1.45;
    ">
        <b>global_row_id:</b> {html.escape(str(row["global_row_id"]))}<br>
        <b>post_id:</b> {html.escape(str(row["post_id"]))}<br>
        <b>split:</b> {html.escape(str(row["split"]))}<br>
        <b>classifier confusion_type:</b> {html.escape(str(row["confusion_type"]))}<br>
        <b>MIL confusion_type:</b> {html.escape(str(row["mil_confusion_type"]))}<br>
        <b>issue:</b> {html.escape(str(row["issue"]))}<br>
        <b>pooling:</b> {html.escape(str(row["pooling_mode"]))} |
        <b>top_k:</b> {html.escape(str(row["top_k"]))} |
        <b>span_lengths:</b> {html.escape(str(row["span_lengths_label"]))} |
        <b>stride:</b> {html.escape(str(row["stride"]))} |
        <b>freeze_encoder:</b> {html.escape(str(row["freeze_encoder"]))}<br>
        <b>selected spans:</b> {html.escape(str(row["selected_span_texts"]))}<br>
        <b>p_original:</b> {float(row["p_inappropriate_original"]):.4f}<br>
        <b>p_masked:</b> {float(row["p_inappropriate_masked"]):.4f}<br>
        <b>prob_drop:</b> {float(row["prob_drop"]):.4f}<br>
        <b>MIL probability:</b> {float(row["mil_probability"]):.4f}<br>
        <b>masked_word_ratio:</b> {float(row["masked_word_ratio"]):.4f}<br>
        <hr>
        <p>{marked}</p>
    </div>
    """))

In [45]:
test_valid = test_argument_df.copy()

best_examples = (
    test_valid
    .sort_values("prob_drop", ascending=False)
    .reset_index(drop=True)
)

print("Best MIL perturbation examples:")
display(
    best_examples[
        [
            "global_row_id",
            "post_id",
            "confusion_type",
            "mil_confusion_type",
            "prob_drop",
            "masked_word_ratio",
            "selected_span_texts",
        ]
    ].head(10)
)

for i in range(min(5, len(best_examples))):
    show_mil_example(best_examples, index=i)

Best MIL perturbation examples:


,global_row_id,post_id,confusion_type,mil_confusion_type,prob_drop,masked_word_ratio,selected_span_texts
0,2111,1769,FP,FP,0.992804,0.441176,[causes obesity. I'm not going to go for a run...
1,1803,250,FP,FP,0.991283,0.579710,[white anglo-saxon protestants were systematic...
2,1835,407,FP,FP,0.991106,0.530303,[weren't going to abide by it? And... the answ...
3,2085,1632,TP,TP,0.982897,0.297030,[add to their family. DO NOT let anyone tell y...
4,1813,291,TP,FN,0.970108,0.294118,[intelligence.....if we were created. If we ha...
5,2009,1249,FP,TN,0.969126,0.400000,"[open too. Very poor people skills, including ..."
6,2022,1289,FP,FP,0.965048,0.368421,[like about 1 hr just to find their pants or s...
7,1935,887,FP,FP,0.961264,0.321101,[I'm sure!!!!!! Most of them couldn't care les...
8,1762,45,FP,FP,0.954114,0.615385,[change. Who says that he will be lousy foreve...
9,1913,762,TP,TP,0.939363,0.432836,[inferior which then justifies locking them up...


In [46]:
worst_examples = (
    test_valid
    .sort_values("prob_drop", ascending=True)
    .reset_index(drop=True)
)

print("Lowest or negative MIL perturbation examples:")
display(
    worst_examples[
        [
            "global_row_id",
            "post_id",
            "confusion_type",
            "mil_confusion_type",
            "prob_drop",
            "masked_word_ratio",
            "selected_span_texts",
        ]
    ].head(10)
)

for i in range(min(5, len(worst_examples))):
    show_mil_example(worst_examples, index=i)

Lowest or negative MIL perturbation examples:


,global_row_id,post_id,confusion_type,mil_confusion_type,prob_drop,masked_word_ratio,selected_span_texts
0,1786,167,TN,TN,-0.995522,0.775862,[felt like killing somone it may be a little d...
1,1766,63,TN,TN,-0.994159,0.694444,"[right, killing someone and getting away with ..."
2,2080,1618,TN,TN,-0.993370,0.593220,[and she tends to make more mistakes even now ...
3,2005,1217,TN,TN,-0.992932,0.241935,"[well aware that there is no such thing as a, ..."
4,1787,168,TN,FP,-0.991526,0.291667,"[spelling skills. Plus it's more, it. Plus you..."
5,2144,1939,FN,FN,-0.990132,0.769231,"[punishing could last up to, I don't think tha..."
6,1947,950,TN,TN,-0.988893,1.176471,[murder and told me about it. I wouldnt turn h...
7,1929,859,TN,TN,-0.988210,0.810811,"[Personal pursuit is better in, better yoursel..."
8,2083,1626,TN,TN,-0.986880,0.540541,[not kill me? Everyone wants to know what happ...
9,2081,1621,TN,TN,-0.982902,0.833333,"[spank the child as long, It's ok for the pare..."
